# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jerovernay/FlyRank-Internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

Lane 3 — **Structured Content Archetype Clustering**. Unsupervised grouping of pages from safe
metric signals, archetypes named only after inspecting centroids, each mapped to an action, then
compared against the Week-4 rule baseline (ML-07) on an outcome window neither approach saw.

## 1. Method choice and why

**Method: K-Means** (k chosen below), on standardised numeric features, with PCA used only as a
2-D projection for looking at the result — never as a preprocessing step feeding the fit, so the
centroids stay readable in original units.

**Why clustering at all.** Lane 3's question is *"what performance archetypes exist across the
inventory?"* — that is structure, not a target. ML-03 confirmed there is no archetype column
anywhere in the schema, so there is nothing to supervise against. Naming happens *after* looking
at centroids, which is the lane's stated #1 mistake to avoid.

**Why plain K-Means and not something bigger.** Considered and rejected on measured evidence,
not assumption:
- *Approximate nearest neighbours (ANN / FAISS / HNSW)* — the wrong tool by construction.
  K-Means compares each point to `k` centroids, not to every other point, so there is no
  nearest-neighbour search to approximate. ANN would optimise a query this method never issues.
- *MiniBatchKMeans* — timed against plain `KMeans` at our real N below. Both finish in seconds,
  so the deterministic option wins. Complexity has to be earned.

The genuine scaling problem turned out to be elsewhere: `silhouette_score` is O(N²) in pair
distances, so it is evaluated on a fixed 10,000-row subsample throughout (subsample noise
measured at ±0.011, negligible against the effects we care about).

**Features — four pure search-side signals**, all aggregated from `month=2026-03` only:

| Feature | Why |
|---|---|
| `log_impressions` | heavy-tailed; log so volume doesn't dominate the scaler |
| `log_clicks` | same |
| `ctr` | clicks / impressions × 100 (these are ×100 percentages, per the data dictionary) |
| `avg_position` | impression-weighted; rows with no position data are excluded, never zero-filled |

**What was dropped, and why it matters.** The first feature set also carried `word_count`,
`has_word_count`, `has_ga4_coverage`, and one-hot `content_type` / `main_intent`. It produced a
k=2 solution that a **single depth-1 rule reproduces at 99.2% accuracy:
`has_word_count`**. The clustering was separating *pages we have metadata about* from *pages we
do not* — a data-availability artifact wearing the costume of an archetype. Our own
`fillna(0)` caused it: median `word_count` is ~2,780, so filling missing with 0 manufactures a
2,780-unit gap that dwarfs every real signal after scaling. Coverage is also client-structured
(35 of 44 clients have `word_count` ~100% present, 9 partial, none fully missing), so the flag
partly encodes *client identity* — the one thing that must never define an archetype.

Dropping `engaged_sessions` was the second pass: with it in, one cluster came back at
`ga4_coverage = 1.000` exactly. GA4 reaches only 27.3% of content (ML-04), so that feature
re-introduced the same coverage axis through a different door. Four search-side features are
fully populated for every page in the population, which also dissolves the standing caveat that
clustering was dropping ~30% of pages.

In [1]:
import os, time, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", message=".*valid feature names.*")

SEED, VOL_FLOOR, N_FOLDS, K_CLUSTERS = 42, 100, 5, 4
FEATURES = ["log_impressions", "log_clicks", "ctr", "avg_position"]
OUTD = "../outputs"
os.makedirs(OUTD, exist_ok=True)

MARCH_CACHE = f"{OUTD}/w05_march_features.parquet"
APRIL_CACHE = f"{OUTD}/w05_april_outcome.parquet"


def _hf_con():
    """DuckDB + HF secret. Token from a local gitignored .env or the environment - never a cell."""
    import duckdb
    if "HF_TOKEN" not in os.environ and os.path.exists("../../.env"):
        for line in open("../../.env"):
            if line.startswith("HF_TOKEN"):
                os.environ["HF_TOKEN"] = line.strip().split("=", 1)[1]
    con = duckdb.connect()
    con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{os.environ['HF_TOKEN']}')")
    return con


BASE = "hf://datasets/FlyRank/internship-warehouse"

if os.path.exists(MARCH_CACHE):
    march = pd.read_parquet(MARCH_CACHE)
else:
    con = _hf_con()
    march = con.sql(f"""
        WITH gsc_agg AS (
            SELECT content_hash_id,
                   ANY_VALUE(client_hash_id)      AS client_hash_id,
                   COUNT(DISTINCT client_hash_id) AS n_clients,
                   SUM(gsc_impressions)           AS total_impressions,
                   SUM(gsc_clicks)                AS total_clicks,
                   SUM(gsc_impressions * gsc_avg_position)
                     / NULLIF(SUM(gsc_impressions), 0) AS avg_position
            FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
            GROUP BY content_hash_id
        ),
        ga4_agg AS (
            SELECT content_hash_id, SUM(ga4_engaged_sessions) AS engaged_sessions
            FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-03/*.parquet')
            WHERE ga4_data_available IS TRUE GROUP BY content_hash_id
        )
        SELECT g.content_hash_id, g.client_hash_id, g.n_clients,
               g.total_impressions, g.total_clicks, g.avg_position,
               COALESCE(a.engaged_sessions, 0) AS engaged_sessions,
               (a.content_hash_id IS NOT NULL) AS has_ga4_coverage,
               dc.word_count, dc.content_type, dc.main_intent, dc.is_published, dc.is_deleted
        FROM gsc_agg g
        LEFT JOIN ga4_agg a USING (content_hash_id)
        LEFT JOIN read_parquet('{BASE}/dim_content.parquet') dc USING (content_hash_id)
    """).df()
    march.to_parquet(MARCH_CACHE, index=False)

# Population: EXACTLY the ML-07 baseline's filters, so rule and model score the same pages.
pop = march[march.is_published & (~march.is_deleted)
            & (march.total_impressions > 0) & (march.avg_position > 0)
            & (march.total_impressions >= VOL_FLOOR)].copy().reset_index(drop=True)
pop["ctr"] = pop.total_clicks / pop.total_impressions * 100

X = pd.DataFrame({"log_impressions": np.log1p(pop.total_impressions),
                  "log_clicks": np.log1p(pop.total_clicks),
                  "ctr": pop.ctr,
                  "avg_position": pop.avg_position})[FEATURES]

print(f"March rows: {len(march)} | modelling population: {len(pop)} pages"
      f" / {pop.client_hash_id.nunique()} clients")
print("feature matrix:", X.shape, "| NaNs:", int(X.isna().sum().sum()))

March rows: 331437 | modelling population: 101409 pages / 44 clients
feature matrix: (101409, 4) | NaNs: 0


In [2]:
# Evidence for the two method choices claimed above.
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.tree import DecisionTreeClassifier

Xs_all = StandardScaler().fit_transform(X)

t0 = time.time(); KMeans(n_clusters=5, random_state=SEED, n_init=10).fit(Xs_all); t_km = time.time()-t0
t0 = time.time(); MiniBatchKMeans(n_clusters=5, random_state=SEED, n_init=10).fit(Xs_all); t_mb = time.time()-t0
print(f"timing at n={len(X)}: KMeans {t_km:.1f}s | MiniBatchKMeans {t_mb:.1f}s"
      f"  -> both trivial, keep deterministic KMeans")

# The rejected feature set, reproduced: is one binary flag doing all the work?
rej = pd.concat([X,
                 pop.word_count.notna().astype(int).rename("has_word_count"),
                 pop.word_count.fillna(0).rename("word_count"),
                 pop.has_ga4_coverage.astype(int).rename("has_ga4_coverage"),
                 np.log1p(pop.engaged_sessions).rename("log_engaged_sessions")], axis=1)
km_rej = KMeans(n_clusters=2, random_state=SEED, n_init=10).fit(StandardScaler().fit_transform(rej))
t1 = DecisionTreeClassifier(max_depth=1, random_state=SEED).fit(rej, km_rej.labels_)
print(f"\nREJECTED feature set, k=2: a single depth-1 rule on "
      f"'{rej.columns[t1.tree_.feature[0]]}' reproduces the clusters at "
      f"{t1.score(rej, km_rej.labels_):.4f} accuracy -> data-availability artifact, not an archetype.")

cov = pop.groupby("client_hash_id").word_count.apply(lambda s: s.notna().mean())
print(f"word_count coverage is client-structured: {(cov > 0.95).sum()} clients ~fully present, "
      f"{((cov >= 0.05) & (cov <= 0.95)).sum()} partial, {(cov < 0.05).sum()} fully missing "
      f"(of {len(cov)})")

timing at n=101409: KMeans 2.7s | MiniBatchKMeans 1.4s  -> both trivial, keep deterministic KMeans



REJECTED feature set, k=2: a single depth-1 rule on 'has_word_count' reproduces the clusters at 0.9917 accuracy -> data-availability artifact, not an archetype.
word_count coverage is client-structured: 35 clients ~fully present, 9 partial, 0 fully missing (of 44)


## 2. Split design

**Grouped by client, not by time.** This is structural clustering, so there is no future to leak
into — the risk is **client leakage**. If one client's pages dominate a cluster we have learned
that client's house style, not a cross-client archetype (ML-03 flagged exactly this:
*"a cluster that only exists for one client isn't an archetype, it's that client"*).
`client_hash_id` is a grouping key only and never enters the feature matrix.

**Why `GroupKFold` rather than one held-out split.** A single 80/20 client split is a lottery
here. There are only 44 clients in the population and their sizes are wildly uneven, so
`GroupShuffleSplit(test_size=0.20)` holds out 20% of *clients* — which landed anywhere from
**0.5% to 29.6% of pages** depending on the seed. At seed 42 the held-out set was also 100%
`keyword article` and 89.6% word-count-covered against 70.1% in train: not a random sample of the
inventory, a different mix. Choosing a seed that produced a comfortable split would be
seed-shopping against the test set. `GroupKFold` removes the choice — every client is held out
exactly once, and every page is scored by a model that never saw its client.

One client alone is **21% of the whole population**, which fold 0 isolates automatically. That
concentration is a real property of this panel and is carried into the limitations.

**Pre-declared success definition** (written before the outcome window was opened):

> A page **improved** if its April CTR exceeded its March CTR, requiring April impressions ≥ 100
> so the April CTR is not noise.

**Correction, stated openly.** The definition originally pre-declared was *"April CTR reaches
the March median CTR of its own position tier."* That bar is **degenerate**: the March tier
medians are `top_3` 0.211, `page_1` 0.190, `striking` 0.101, but `page_3_5` **0.000** and `deep`
**0.000** — so every page in the bottom two tiers passes for free, inflating the base rate to
0.515 and making the rule look worse than random. The rule can only ever flag pages in the top
three tiers, so it was being graded against a real bar while the comparison pool was padded with
free passes. The directional definition above replaces it as primary; the tier-median definition
is retained as a secondary, restricted to the three tiers where the bar is actually positive.
Swapping a metric after seeing results is only legitimate if you say so — so it is said here.

In [3]:
from sklearn.model_selection import GroupKFold

# Grain probe: the group split is only clean if content -> client is 1:1
assert (march.n_clients > 1).sum() == 0, "a content item maps to >1 client; grouping is unsafe"
assert march.content_hash_id.duplicated().sum() == 0
print("PASS grain probe: content_hash_id -> client_hash_id is 1:1, no duplicates")

groups = pop.client_hash_id.values
folds = list(GroupKFold(n_splits=N_FOLDS).split(X, groups=groups))
for i, (tr, te) in enumerate(folds):
    assert not (set(pop.iloc[tr].client_hash_id) & set(pop.iloc[te].client_hash_id))
    print(f"fold {i}: train {len(tr):>6}p/{pop.iloc[tr].client_hash_id.nunique():>2}c "
          f"| held-out {len(te):>6}p/{pop.iloc[te].client_hash_id.nunique():>2}c")
print("PASS no client appears in both sides of any fold")

# Why a single grouped split was rejected: page-share lottery across seeds
from sklearn.model_selection import GroupShuffleSplit
shares = []
for s in [0, 1, 7, 42, 2026]:
    _, t = next(GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=s).split(X, groups=groups))
    shares.append(f"seed {s}: {len(t)/len(pop):.1%}")
print("\nsingle 80/20 grouped split would hold out -> " + " | ".join(shares))

# Leakage guards
assert "client_hash_id" not in X.columns and "content_hash_id" not in X.columns
assert not any(c in X.columns for c in ["trend_pct", "trend_direction", "content_updated_date"])
assert not any(c.startswith("april") for c in X.columns)
print("\nPASS client/content ids are not features")
print("PASS trend_pct / trend_direction / content_updated_date excluded (label trap + live field)")
print("PASS no April column is a feature - the outcome window is opened once, for grading only")
print("PASS population uses month=2026-03 only; the _sample table (sealed June) is never read")

PASS grain probe: content_hash_id -> client_hash_id is 1:1, no duplicates


fold 0: train  79776p/43c | held-out  21633p/ 1c


fold 1: train  81425p/40c | held-out  19984p/ 4c
fold 2: train  81478p/30c | held-out  19931p/14c


fold 3: train  81478p/31c | held-out  19931p/13c


fold 4: train  81479p/32c | held-out  19930p/12c
PASS no client appears in both sides of any fold



single 80/20 grouped split would hold out -> seed 0: 14.7% | seed 1: 0.5% | seed 7: 29.6% | seed 42: 6.9% | seed 2026: 3.4%

PASS client/content ids are not features
PASS trend_pct / trend_direction / content_updated_date excluded (label trap + live field)
PASS no April column is a feature - the outcome window is opened once, for grading only
PASS population uses month=2026-03 only; the _sample table (sealed June) is never read


## 3. Train + compare vs my baseline

**Choosing k.** Silhouette alone cannot pick k here. It rewards degenerate solutions: on one
client subset a k=2 split scoring 0.693 turned out to be a 1,040-page pocket against everything
else. So the sweep runs inside each fold's training clients and any k whose smallest cluster
falls below **2% of pages** is rejected outright, with the winner taken on mean silhouette
across folds and confirmed by reading the profile.

In [4]:
from sklearn.metrics import silhouette_score

MIN_SIZE_FRAC = 0.02
rows = []
for k in range(2, 11):
    sils, mins = [], []
    for tr, _ in folds:
        sc = StandardScaler().fit(X.iloc[tr]); Xs = sc.transform(X.iloc[tr])
        km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(Xs)
        sils.append(silhouette_score(Xs, km.labels_, sample_size=10_000, random_state=SEED))
        mins.append(np.bincount(km.labels_, minlength=k).min() / len(tr))
    rows.append({"k": k, "sil_mean": np.mean(sils), "sil_sd": np.std(sils),
                 "min_size_frac": min(mins), "passes_guard": min(mins) >= MIN_SIZE_FRAC})
sweep = pd.DataFrame(rows)
print(sweep.round(4).to_string(index=False))
elig_k = sweep[sweep.passes_guard]
print(f"\neligible k: {elig_k.k.tolist()} | best mean silhouette at k="
      f"{int(elig_k.loc[elig_k.sil_mean.idxmax(), 'k'])}")

 k  sil_mean  sil_sd  min_size_frac  passes_guard
 2    0.3677  0.0083         0.3760          True
 3    0.3660  0.0112         0.1189          True
 4    0.3874  0.0118         0.0591          True
 5    0.3235  0.0117         0.0503          True
 6    0.3385  0.0120         0.0201          True
 7    0.3427  0.0066         0.0194         False
 8    0.3273  0.0079         0.0189         False
 9    0.3202  0.0084         0.0023         False
10    0.3161  0.0044         0.0016         False

eligible k: [2, 3, 4, 5, 6] | best mean silhouette at k=4


In [5]:
from scipy.optimize import linear_sum_assignment

# Out-of-fold cluster assignment. Each fold's model is aligned to fold 0's centroids
# (Hungarian matching in a common comparison space) so the labels mean the same thing.
ref_scaler = StandardScaler().fit(X)
labels = np.full(len(X), -1); dists = np.full(len(X), np.nan); ref = None; align_cost = []
for tr, te in folds:
    sc = StandardScaler().fit(X.iloc[tr])
    km = KMeans(n_clusters=K_CLUSTERS, random_state=SEED, n_init=10).fit(sc.transform(X.iloc[tr]))
    cent = ref_scaler.transform(pd.DataFrame(sc.inverse_transform(km.cluster_centers_), columns=FEATURES))
    if ref is None:
        ref, mapping = cent, np.arange(K_CLUSTERS)
    else:
        d = np.linalg.norm(cent[:, None, :] - ref[None, :, :], axis=2)
        r, c = linear_sum_assignment(d); mapping = np.empty(K_CLUSTERS, int); mapping[r] = c
        align_cost.append(round(d[r, c].mean(), 3))
    Xte = sc.transform(X.iloc[te]); raw = km.predict(Xte)
    labels[te] = mapping[raw]
    dists[te] = np.linalg.norm(Xte - km.cluster_centers_[raw], axis=1)

pop["cluster"] = labels; pop["centroid_dist"] = dists
assert (pop.cluster >= 0).all(), "every page must receive an out-of-fold label"
print(f"centroid alignment cost vs fold 0 (std-units, lower = more stable): {align_cost}")

g = pop.groupby("cluster")
prof = g.agg(n=("content_hash_id", "size"), impressions=("total_impressions", "median"),
             clicks=("total_clicks", "median"), ctr=("ctr", "median"),
             position=("avg_position", "median"))
prof["pct"] = (prof.n / len(pop) * 100).round(1)
prof["pct_zero_clicks"] = g.total_clicks.apply(lambda s: (s == 0).mean()).round(3)
# descriptors only - these were NOT clustering inputs
prof["ga4_cov"] = g.has_ga4_coverage.mean().round(3)
prof["word_count"] = g.word_count.median()
prof["n_clients"] = g.client_hash_id.nunique()
prof["top_client_share"] = g.client_hash_id.apply(lambda s: s.value_counts(normalize=True).iloc[0]).round(3)
print("\nOut-of-fold centroid profile (medians, original units):")
print(prof.round(3).to_string())

centroid alignment cost vs fold 0 (std-units, lower = more stable): [np.float64(0.084), np.float64(0.296), np.float64(0.202), np.float64(0.219)]

Out-of-fold centroid profile (medians, original units):
             n  impressions  clicks    ctr  position   pct  pct_zero_clicks  ga4_cov  word_count  n_clients  top_client_share
cluster                                                                                                                      
0         6546        498.5     6.0  1.172     6.352   6.5            0.000    0.798      2757.0         36             0.154
1        30236       4388.5    10.0  0.277     5.736  29.8            0.001    0.727      2771.0         30             0.260
2        52062        469.0     0.0  0.000     8.225  51.3            0.534    0.460      2775.0         43             0.207
3        12565        260.0     0.0  0.000    41.422  12.4            0.792    0.380      2812.0         31             0.195


### The archetypes, named after reading the profile

k=4 is the silhouette peak among the k that pass the min-size guard (0.387, about 1.7 sd clear of
the runner-up). Reading the profile top to bottom:

| cluster | archetype | what the numbers say | action |
|---|---|---|---|
| 0 | **Overlooked** | ~500 impressions but CTR 1.17% — three times anyone else — and *never* zero clicks. Small audience, converts unusually well. | improve / expand |
| 1 | **Steady performers** | ~4,400 impressions, position 5.7, reliable clicks, unremarkable CTR. The working core of the inventory. | protect / monitor |
| 2 | **Long tail** | ~470 impressions, zero clicks at the median, mid position. Half the inventory sits here. | monitor |
| 3 | **Buried** | position 41, 79% never clicked. Effectively invisible in search. | prune / rewrite |

A fifth archetype appeared in an earlier run — high volume, poor position, "ranking lags demand".
It was the cluster sitting at `ga4_coverage = 1.000`, and it stopped existing once
`engaged_sessions` was removed. It was a measurement artifact, not a kind of page.

The honest cost of that: **there is now no "ranks below its demand" archetype**, because cluster 1
carries both high volume *and* good position. That is arguably the most commercially actionable
group, and this feature set cannot isolate it. Carried to Limitations.

In [6]:
ARCHETYPES = {0: "Overlooked", 1: "Steady performers", 2: "Long tail", 3: "Buried"}
ACTIONS = {"Overlooked": "improve/expand", "Steady performers": "protect/monitor",
           "Long tail": "monitor", "Buried": "prune/rewrite"}
pop["archetype"] = pop.cluster.map(ARCHETYPES)
pop["archetype_action"] = pop.archetype.map(ACTIONS)

# --- The ML-07 rule, recomputed in THIS run (not read from the old CSV) ---
POS_BINS = [-1, 3, 10, 20, 50, 10_000]
POS_LABELS = ["top_3", "page_1", "striking", "page_3_5", "deep"]
pop["pos_tier"] = pd.cut(pop.avg_position, bins=POS_BINS, labels=POS_LABELS)

# Reproduction check against ML-07 (full-population peer medians, as ML-07 computed them)
peer_full = pop.groupby("pos_tier", observed=True)["ctr"].median()
gap_full = pop.pos_tier.map(peer_full).astype(float) - pop.ctr
score_full = np.where(gap_full > 0, pop.total_impressions * gap_full / 100, 0.0)
if os.path.exists(f"{OUTD}/baseline_action_score.csv"):
    old = pd.read_csv(f"{OUTD}/baseline_action_score.csv")[["content_hash_id", "score"]]
    m = pop.assign(s=score_full).merge(old, on="content_hash_id")
    print(f"ML-07 reproduction: {len(m)} shared rows, score agreement "
          f"{np.isclose(m.s, m.score, atol=0.01).mean():.4f}")

# Fold-honest version: peer medians come from each fold's TRAINING clients only (guard #5)
pop["peer_median_ctr"] = np.nan
for tr, te in folds:
    med = pop.iloc[tr].groupby("pos_tier", observed=True)["ctr"].median()
    pop.loc[pop.index[te], "peer_median_ctr"] = pop.iloc[te].pos_tier.map(med).astype(float).values
gap = pop.peer_median_ctr - pop.ctr
pop["baseline_score"] = np.where(gap > 0, pop.total_impressions * gap / 100, 0.0)
pop["baseline_action"] = np.where(gap > 0, "snippet_fix", "monitor")
print(f"rule flags snippet_fix on {(pop.baseline_action == 'snippet_fix').sum()} pages "
      f"({(pop.baseline_action == 'snippet_fix').mean():.1%})")

ML-07 reproduction: 101409 shared rows, score agreement 1.0000


rule flags snippet_fix on 44857 pages (44.2%)


In [7]:
# --- The outcome window: April is opened ONCE, here, for grading only ---
if os.path.exists(APRIL_CACHE):
    april = pd.read_parquet(APRIL_CACHE)
else:
    april = _hf_con().sql(f"""
        SELECT content_hash_id,
               SUM(gsc_impressions) AS april_impressions,
               SUM(gsc_clicks)      AS april_clicks,
               SUM(gsc_impressions * gsc_avg_position)
                 / NULLIF(SUM(gsc_impressions), 0) AS april_avg_position
        FROM read_parquet('{BASE}/fact_content_daily_performance/month=2026-04/*.parquet')
        GROUP BY content_hash_id""").df()
    april.to_parquet(APRIL_CACHE, index=False)

pop = pop.merge(april, on="content_hash_id", how="left")
pop["april_ctr"] = np.where(pop.april_impressions > 0,
                            pop.april_clicks / pop.april_impressions * 100, np.nan)

elig = pop[pop.april_impressions >= VOL_FLOOR].copy()
elig["improved"] = elig.april_ctr > elig.ctr                       # PRIMARY, directional
strict_ok = elig.peer_median_ctr > 0                               # bar must be positive
elig["improved_strict"] = np.where(strict_ok, elig.april_ctr >= elig.peer_median_ctr, np.nan)

print(f"March population {len(pop)} -> eligible {len(elig)} ({len(elig)/len(pop):.1%}); "
      f"{int((pop.april_impressions.fillna(0) < VOL_FLOOR).sum())} fell below the April volume floor")
print("attrition is NOT scored as failure - those pages are excluded and counted here")
base = elig.improved.mean(); base_strict = elig.loc[strict_ok, "improved_strict"].mean()
print(f"\nBASE RATE  directional={base:.4f} | strict(restricted, n={int(strict_ok.sum())})={base_strict:.4f}")

March population 101409 -> eligible 88474 (87.2%); 12935 fell below the April volume floor
attrition is NOT scored as failure - those pages are excluded and counted here

BASE RATE  directional=0.2925 | strict(restricted, n=75020)=0.4277


In [8]:
# --- (a) SET-LEVEL comparison table ---
rows = [{"group": "ALL eligible (base rate)", "n": len(elig), "improved": base, "lift": 1.0,
         "strict": base_strict, "march_ctr": elig.ctr.median()}]
r = elig[elig.baseline_action == "snippet_fix"]
rows.append({"group": "RULE: snippet_fix (ML-07)", "n": len(r), "improved": r.improved.mean(),
             "lift": r.improved.mean()/base,
             "strict": r.loc[r.peer_median_ctr > 0, "improved_strict"].mean(),
             "march_ctr": r.ctr.median()})
for a in ARCHETYPES.values():
    s = elig[elig.archetype == a]; st = s.loc[s.peer_median_ctr > 0, "improved_strict"]
    rows.append({"group": f"MODEL: {a} ({ACTIONS[a]})", "n": len(s), "improved": s.improved.mean(),
                 "lift": s.improved.mean()/base, "strict": st.mean() if len(st) else np.nan,
                 "march_ctr": s.ctr.median()})
print("=== (a) MODEL vs BASELINE, same population, same April window ===")
print(pd.DataFrame(rows).round(4).to_string(index=False))

# --- (b) head-to-head at matched K ---
act = elig[elig.archetype == "Overlooked"].sort_values("centroid_dist")
print("\n=== (b) precision@K, directional (model ranked by distance to its centroid) ===")
print(f"{'K':>6} {'rule':>8} {'model':>8} {'base':>8}")
for K in (100, 500, 1000, 5000):
    rk = elig.nlargest(K, "baseline_score").improved.mean()
    mk = act.head(K).improved.mean() if len(act) >= K else np.nan
    print(f"{K:>6} {rk:>8.4f} {mk:>8.4f} {base:>8.4f}")

=== (a) MODEL vs BASELINE, same population, same April window ===
                                     group     n  improved   lift  strict  march_ctr
                  ALL eligible (base rate) 88474    0.2925 1.0000  0.4277     0.1483
                 RULE: snippet_fix (ML-07) 36704    0.3335 1.1404  0.2103     0.0000
        MODEL: Overlooked (improve/expand)  5715    0.1538 0.5259  0.8161     1.1678
MODEL: Steady performers (protect/monitor) 30185    0.3277 1.1205  0.5687     0.2768
                MODEL: Long tail (monitor) 42911    0.3108 1.0627  0.3026     0.0406
             MODEL: Buried (prune/rewrite)  9663    0.1830 0.6256  0.1977     0.0000



=== (b) precision@K, directional (model ranked by distance to its centroid) ===
     K     rule    model     base
   100   0.4500   0.1300   0.2925
   500   0.4240   0.1380   0.2925
  1000   0.4350   0.1300   0.2925


  5000   0.4474   0.1560   0.2925


**Read (a) and (b) together and they disagree — which is the finding.**

On the directional metric the rule wins (0.334 vs a 0.293 base) and Overlooked looks poor (0.154).
On the strict metric Overlooked wins overwhelmingly (0.816 vs 0.428) and the rule *loses* (0.210).
Same pages, same month, opposite conclusions.

Neither is a fair comparison, because **both metrics are confounded by March CTR** — and in
opposite directions. The rule selects pages with median March CTR of **0.000**; Overlooked has
median **1.168**. Directional improvement is mechanically easier from a low base, and the strict
"already above your tier median" bar is mechanically easier from a high one. The next cell
measures the confound and then removes it.

In [9]:
# --- (c) the confound, measured ---
print("=== improve-rate by March CTR (the confound, directional metric) ===")
elig["stratum"] = np.where(elig.ctr == 0, "zero", "")
nz = elig.ctr > 0
elig.loc[nz, "stratum"] = pd.qcut(elig.loc[nz, "ctr"], 8, duplicates="drop").astype(str)
strata = elig.groupby("stratum").agg(n=("improved", "size"), march_ctr=("ctr", "median"),
                                     base_improved=("improved", "mean"))
print(strata.sort_values("march_ctr").round(4).to_string())
elig["exp"] = elig.stratum.map(strata.base_improved)

# --- (d) confound removed: indirect standardisation, client-clustered bootstrap CI ---
rng = np.random.default_rng(SEED)
clients = elig.client_hash_id.unique()
by_client = {c: gg for c, gg in elig.groupby("client_hash_id")}

def oe(idx, name, B=300):
    gsub = elig.loc[idx]
    if len(gsub) == 0: return None
    point = gsub.improved.mean() / gsub["exp"].mean()
    boots = []
    for _ in range(B):
        s = pd.concat([by_client[c] for c in rng.choice(clients, len(clients), replace=True)])
        s = s[s.index.isin(gsub.index)]
        if len(s) > 30 and s["exp"].mean() > 0:
            boots.append(s.improved.mean() / s["exp"].mean())
    lo, hi = np.percentile(boots, [2.5, 97.5])
    return {"group": name, "n": len(gsub), "O/E": point, "lo95": lo, "hi95": hi,
            "beats_chance": "yes" if lo > 1 else ("WORSE" if hi < 1 else "no")}

res = [oe(elig.index[elig.baseline_action == "snippet_fix"], "RULE: snippet_fix")]
for a in ARCHETYPES.values():
    res.append(oe(elig.index[elig.archetype == a], f"MODEL: {a}"))
for K in (500, 1000):
    res.append(oe(elig.nlargest(K, "baseline_score").index, f"RULE top-{K}"))
    res.append(oe(elig[elig.archetype == "Overlooked"].nsmallest(K, "centroid_dist").index,
                  f"MODEL Overlooked top-{K}"))
print("\n=== (d) CTR-STANDARDISED: observed / expected, 95% CI bootstrapped BY CLIENT ===")
print("O/E > 1 means the group beat what its own starting-CTR mix already predicts")
print(pd.DataFrame([x for x in res if x]).round(4).to_string(index=False))

=== improve-rate by March CTR (the confound, directional metric) ===
                        n  march_ctr  base_improved
stratum                                            
zero                28067     0.0000         0.2609
(-0.000259, 0.085]   7551     0.0563         0.4960
(0.085, 0.14]        7560     0.1128         0.4013
(0.14, 0.201]        7543     0.1696         0.3508
(0.201, 0.274]       7550     0.2356         0.3105
(0.274, 0.37]        7550     0.3180         0.2868
(0.37, 0.498]        7551     0.4367         0.2205
(0.498, 0.755]       7551     0.6004         0.2217
(0.755, 13.527]      7551     1.0360         0.1694



=== (d) CTR-STANDARDISED: observed / expected, 95% CI bootstrapped BY CLIENT ===
O/E > 1 means the group beat what its own starting-CTR mix already predicts
                    group     n    O/E   lo95   hi95 beats_chance
        RULE: snippet_fix 36704 1.0319 0.9248 1.1642           no
        MODEL: Overlooked  5715 0.8956 0.7137 1.0499           no
 MODEL: Steady performers 30185 1.0590 0.8653 1.2332           no
         MODEL: Long tail 42911 1.0409 0.9296 1.1501           no
            MODEL: Buried  9663 0.6452 0.4560 0.8099        WORSE
             RULE top-500   500 0.9254 0.8498 1.1901           no
 MODEL Overlooked top-500   500 0.8147 0.6103 1.0634           no
            RULE top-1000  1000 0.9640 0.9018 1.2292           no
MODEL Overlooked top-1000  1000 0.7675 0.6402 0.9115        WORSE


## 4. Errors and interpretation

**The headline, stated plainly: once starting CTR is controlled for, neither the ML-07 rule nor
the archetype clustering demonstrates skill at predicting April CTR movement.** Every positive
result has a confidence interval straddling 1.0:

| group | O/E | 95% CI | verdict |
|---|---|---|---|
| RULE: snippet_fix | 1.032 | 0.925 – 1.164 | indistinguishable from chance |
| Steady performers | 1.059 | 0.865 – 1.233 | indistinguishable |
| Long tail | 1.041 | 0.930 – 1.150 | indistinguishable |
| Overlooked | 0.896 | 0.714 – 1.050 | indistinguishable |
| RULE top-500 | 0.925 | 0.850 – 1.190 | indistinguishable |
| **Buried** | **0.645** | **0.456 – 0.810** | **worse than chance** |
| **Overlooked top-1000** | **0.768** | **0.640 – 0.912** | **worse than chance** |

The rule's apparent 1.14× lift was **regression to the mean**: it selects pages whose March CTR is
unusually low for their position tier, and unusually low values drift back up on their own. Its
most confident picks — the top 500 by score — land at 0.925, slightly *below* what similar-CTR
pages achieve unaided. This is the ML-07 caveat (*the rule favours high-traffic pages*) showing up
as a measurable cost rather than a suspicion.

**The two robust findings both point downward, and one is useful.** `Buried` pages recover
markedly *less* than their CTR-matched peers (O/E 0.645). That is real evidence that buried pages
do not self-heal — which is a genuine argument for prune/rewrite over monitor, and the most
decision-relevant result in the notebook. It is also the only place where an archetype earns its
action from measured behaviour rather than from a plausible story about the centroid.

**Where the two approaches disagree.** The rule and the clustering barely address the same pages:
71.3% of the rule's `snippet_fix` queue is `Long tail` — low-volume, zero-click pages the
clustering says to monitor — and the rule flags **zero** `Overlooked` pages, ever. So the
baseline's queue is dominated by the half of the inventory with the least to gain, and it never
looks at the group with the strongest conversion.

**What defines each archetype** (depth-3 tree, ~93% faithful to the cluster labels): clicks first,
then position, then CTR. The rules read as plain sentences — "few clicks, position past 28 →
Buried", "few clicks, decent position, CTR above 0.8 → Overlooked" — which is what a readable
clustering should look like. Permutation importance is not used: it is a supervised diagnostic and
does not transfer to clusters, so a surrogate tree is the honest analogue.

**Limitations.**
- **No causal claim is available or intended.** Nothing here was an intervention; April movement
  is observed, not caused. Regression to the mean affects every group.
- **Attrition is uneven and biases the comparison**: 99.8% of `Steady performers` survive to the
  April volume floor but only 76.9% of `Buried`. The surviving Buried pages are the more visible
  ones, so the true Buried result is probably worse than 0.645, not better.
- **44 clients, one of them 21% of the population.** Client-clustered bootstrap intervals are
  wide for exactly this reason, and that width is honest rather than fixable.
- **GA4 engagement is excluded**, so no archetype describes on-site behaviour. Reintroducing it
  reintroduced a 27.3%-coverage artifact (ML-04), which was the worse trade.
- **No "ranking lags demand" archetype exists** in this feature set — likely the most actionable
  group commercially, and this method cannot isolate it.
- **Clusters are a lens, not labels.** Nothing here licenses treating an archetype as ground truth.

In [10]:
from sklearn.tree import export_text

t3 = DecisionTreeClassifier(max_depth=3, random_state=SEED).fit(X, pop.archetype)
print(f"=== what defines each archetype (surrogate tree, {t3.score(X, pop.archetype):.1%} faithful) ===")
print(export_text(t3, feature_names=FEATURES))

print("=== rule queue composition by archetype ===")
q = elig[elig.baseline_action == "snippet_fix"]
print(pd.DataFrame({"n": q.archetype.value_counts(),
                    "pct_of_queue": (q.archetype.value_counts(normalize=True)*100).round(1)}).to_string())

print("\n=== agreement: archetype x rule action (row %) ===")
print(pd.crosstab(elig.archetype, elig.baseline_action, normalize="index").round(3).to_string())

print("\n=== attrition by archetype (share surviving to the April volume floor) ===")
print(pop.assign(e=pop.april_impressions >= VOL_FLOOR).groupby("archetype")
        .e.agg(["mean", "size"]).round(4).to_string())

=== what defines each archetype (surrogate tree, 93.1% faithful) ===
|--- log_clicks <= 1.50
|   |--- avg_position <= 28.33
|   |   |--- ctr <= 0.80
|   |   |   |--- class: Long tail
|   |   |--- ctr >  0.80
|   |   |   |--- class: Overlooked
|   |--- avg_position >  28.33
|   |   |--- avg_position <= 31.98
|   |   |   |--- class: Buried
|   |   |--- avg_position >  31.98
|   |   |   |--- class: Buried
|--- log_clicks >  1.50
|   |--- ctr <= 0.84
|   |   |--- log_impressions <= 6.94
|   |   |   |--- class: Long tail
|   |   |--- log_impressions >  6.94
|   |   |   |--- class: Steady performers
|   |--- ctr >  0.84
|   |   |--- log_impressions <= 7.80
|   |   |   |--- class: Overlooked
|   |   |--- log_impressions >  7.80
|   |   |   |--- class: Steady performers

=== rule queue composition by archetype ===
                       n  pct_of_queue
archetype                             
Long tail          26177          71.3
Steady performers   7543          20.6
Buried              2984  

In [11]:
# Three concrete wrong cases - what the errors actually look like
a = elig[(elig.archetype == "Steady performers") & (elig.total_impressions > 5000)]
print("[1] 'Steady performers' (action: PROTECT) that lost the most CTR:")
print(a.assign(d=a.ctr-a.april_ctr).nlargest(3, "d")[
    ["content_hash_id","total_impressions","ctr","april_impressions","april_ctr",
     "avg_position","april_avg_position"]].round(3).to_string(index=False))

b = elig[elig.archetype == "Buried"]
print("\n[2] 'Buried' (action: PRUNE/REWRITE) that recovered anyway:")
print(b.assign(g=b.april_ctr-b.ctr).nlargest(3, "g")[
    ["content_hash_id","total_impressions","ctr","avg_position",
     "april_impressions","april_ctr","april_avg_position"]].round(3).to_string(index=False))

c = elig[elig.baseline_action == "snippet_fix"].nlargest(2000, "baseline_score")
print("\n[3] The RULE's highest-scoring picks whose CTR fell furthest:")
print(c.assign(d=c.ctr-c.april_ctr).nlargest(3, "d")[
    ["content_hash_id","total_impressions","ctr","peer_median_ctr","baseline_score",
     "april_impressions","april_ctr","archetype"]].round(3).to_string(index=False))

[1] 'Steady performers' (action: PROTECT) that lost the most CTR:
         content_hash_id  total_impressions   ctr  april_impressions  april_ctr  avg_position  april_avg_position
content_623f10411a2328a8            27564.0 1.223             2648.0      0.038         5.355              10.163
content_ce99990eb0ac05a5             7489.0 1.135              767.0      0.000        10.430              24.046
content_3539d16ff8091e01            11656.0 1.132              911.0      0.000         7.024              10.765

[2] 'Buried' (action: PRUNE/REWRITE) that recovered anyway:
         content_hash_id  total_impressions  ctr  avg_position  april_impressions  april_ctr  april_avg_position
content_17cbb176f31edd35              251.0  0.0        28.454              135.0      4.444              15.904
content_d355c371bb6991eb              199.0  0.0        41.437              107.0      1.869              51.477
content_21f0e3cb5252e840              185.0  0.0        34.146              27

**Why those three are hard.**

1. The `Steady performers` that collapsed all lost *position as well as* CTR — one fell from
   position 5.4 to 10.2 while impressions dropped from 27,564 to 2,648. "Protect" is the right
   action and it still lost, because the archetype describes a page's state, not the competitive
   pressure arriving next month. No March-only feature set can see that coming.
2. The `Buried` pages that recovered went from zero clicks to CTR above 1.8%, and the ones that
   did so mostly moved *up* in position (28.5 → 15.9). So "prune" would have destroyed pages that
   were about to work. Buried is a group-level verdict (O/E 0.645), never a per-page one — which
   is exactly what "clusters are a lens, not labels" means in practice.
3. The rule's worst top picks are all high-volume `Steady performers` whose CTR was mildly below
   tier median (0.139 vs 0.212). The rule multiplies a small CTR gap by a large impression count,
   so a page with 50,236 impressions and a 0.019pp gap outranks a genuinely broken page. That is
   the traffic bias from the ML-07 note, now visible as concrete misranked rows.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

**Reproducibility.** All seeds fixed at 42 (`KMeans`, `GroupKFold` ordering, bootstrap RNG).
Warehouse scans are cached to `work/outputs/*.parquet` (gitignored) and re-pulled from Hugging
Face automatically if absent, so the notebook reproduces from a clean checkout. Tree-ensemble and
K-Means results can shift slightly across library versions; recorded below.

In [12]:
import sklearn, duckdb, sys
print("python  ", sys.version.split()[0])
print("numpy   ", np.__version__)
print("pandas  ", pd.__version__)
print("sklearn ", sklearn.__version__)
print("duckdb  ", duckdb.__version__)
print("\nseeds: KMeans/GroupKFold/bootstrap all fixed at", SEED)

python  

 3.12.10
numpy    2.2.4
pandas   2.2.3
sklearn  1.9.0
duckdb   1.2.1

seeds: KMeans/GroupKFold/bootstrap all fixed at 42
